This file tests all the main functions and functionality before integrating into the main GUI.

4/10/2026 Katelyn: make number of solutions as a user input in the UI

In [1]:
import sys
sys.path.append("C:\\Users\\NikonTE300CE\\Desktop\\automated-sca\\src")

from stage import Stage
from plate import Plate
from chip import Chip
from stepper import *
from arm import *
import serial
import time

In [2]:
arm = Arm("COM5")
zm = arm.zmotor

In [3]:
# calibrate the arm origin
zm.calibrateOrigin()

In [4]:
c = Chip()
p = Plate()
# p = Plate(2,3,40000) # roughly the diameter for 6-well plate, a little off
ser = serial.Serial(port="COM4", baudrate=9600, timeout=0.1) 
s = Stage(p, c, ser)

In [5]:
# move stage to bottom right corner to calibrate origin
s.calibOrigin()

In [6]:
zm.moveToZInUM(-30000)

In [7]:
# get first well position 
b = s.getStageXY() 
print("First well stage position:", b)

firstwellpos = tuple(b)
# save to tuple to do calculations later
print("First well position set to:", firstwellpos)

First well stage position: [-108615, -11216]
First well position set to: (-108615, -11216)


In [8]:
# save current stage position as pen location
pen_spot_pos = s.getStageXY()
print(f"Pen position on stage:, {pen_spot_pos}")

Pen position on stage:, [-48866, -15859]


In [9]:
# user move the stage until the center mark in the camera frame is aligned with the center of the spot from the pen
cam_spot_pos = s.getStageXY()
print(f"Camera spot stage coordinates: {cam_spot_pos}")

Camera spot stage coordinates: [-70352, -4274]


In [10]:
pen_cam_offset_xy = [cam_spot_pos - pen_spot_pos for cam_spot_pos, pen_spot_pos in zip(cam_spot_pos, pen_spot_pos)]
print(f"Pen offset (stage coords) from fiducial marker: {pen_cam_offset_xy}")

Pen offset (stage coords) from fiducial marker: [-21486, 11585]


In [11]:
# Move stage to see first channel on camera, could find the first channel near either A or B on the chip
first_channel = s.getStageXY()
print("First channel stage position:", first_channel)

First channel stage position: [-71667, -3420]


In [12]:
# Channel position calculations
def get_channel_xy(channel_num, first_channel_pos, channel_spacing=260):
    """
    Calculate channel position (camera view)
    channel_num: 1-40
    first_channel_pos: (x, y) calibrated position of channel 1
    """
    offset = (channel_num - 1) * channel_spacing
    x = first_channel_pos[0] - offset  # Channels go in negative X
    y = first_channel_pos[1]
    return (x, y)

def apply_pen_offset(cam_pos, pen_offset_xy):
    """
    Convert camera position to pen position
    """
    return (
        cam_pos[0] - pen_offset_xy[0],
        cam_pos[1] - pen_offset_xy[1]
    )

In [60]:
get_channel_xy(1,first_channel)

(-50524, -11047)

In [4]:
total_channels = 40
num_solutions = 5
# Channel pattern for each solution
def get_channel_pattern(solution_num):
    """
    Returns list of channel numbers for a given solution
    solution_num: 1-6
    Returns: [1,7,13,...] for solution 1, [2,8,14,...] for solution 2, etc.
    """
    channels = []
    for i in range(solution_num, total_channels + 1, num_solutions):
        channels.append(i)
    return channels

In [5]:
get_channel_pattern(1)

[1, 6, 11, 16, 21, 26, 31, 36]

In [14]:
# 1. Setup and calibration data
first_well_pos = firstwellpos  # From calibration
first_channel_pos = first_channel  # From calibration
pen_offset = pen_cam_offset_xy

safe_z = -30000  # microns
in_well_z = -37000
spot_z = -52000

plate_type = "Plate384"
num_solutions = 6
total_channels = 40

# 2. Helper functions
# Channel pattern for each solution
def get_channel_pattern(solution_num):
    """
    Returns list of channel numbers for a given solution
    solution_num: 1-6
    Returns: [1,7,13,...] for solution 1, [2,8,14,...] for solution 2, etc.
    """
    channels = []
    for i in range(solution_num, total_channels + 1, num_solutions):
        channels.append(i)
    return channels

# Well position calculations
def get_well_xy(well_id):
    """
    Calculate well position from first well calibration
    well_id: "A1", "C3", etc.
    first_well_pos: (x, y) calibrated position of A1
    """
    # Determine spacing based on plate
    if plate_type == "Plate384":
        diam = 4500  # microns
    elif plate_type == "Plate96":
        diam = 9000
    # ... etc
    
    row = ord(well_id[0].upper()) - ord('A')  # A=0, B=1, C=2...
    col = int(well_id[1:]) - 1  # 1=0, 2=1, 3=2...
    
    x = first_well_pos[0] - col * diam
    y = first_well_pos[1] - row * diam
    return (x, y)

def get_channel_xy_pen(channel_num):
    # Get camera position then apply pen offset
    cam_pos = get_channel_xy(channel_num, first_channel_pos)
    return apply_pen_offset(cam_pos, pen_offset)

# 3. Main sequence for ONE solution
def print_one_solution(solution_num):
    """Print one solution to its pattern of channels with cleaning at the end"""
    
    # Get well positions
    solution_well_id = f"A{solution_num}"  # A1, A2, ..., A6
    
    solution_well_xy = get_well_xy(solution_well_id)
    
    # Get channels for this solution
    channels = get_channel_pattern(solution_num)
    print(f"Solution {solution_num}: Will print to channels {channels}")
    
    for channel_num in channels:
        print(f"  Processing channel {channel_num}")
        
        # Get pen position for this channel
        channel_xy_pen = get_channel_xy_pen(channel_num)
        
        # 1. Aspirate from solution well
        print(f"    Aspirating from well {solution_well_id}")
        zm.moveToZInUM(safe_z)
        s.moveToPos(solution_well_xy[0], solution_well_xy[1])
        zm.moveToZInUM(in_well_z)
        time.sleep(0.1)  # Brief aspirate
        zm.moveToZInUM(safe_z)
        
        # 2. Dispense to channel
        print(f"    Dispensing to channel {channel_num}")
        s.moveToPos(channel_xy_pen[0], channel_xy_pen[1])
        zm.moveToZInUM(spot_z)
        time.sleep(1)  # Dispense time
        zm.moveToZInUM(safe_z)
    
    # 3. After all channels done, clean in all 6 cleaning wells (C1-C6)
    print(f"Solution {solution_num}: Cleaning in wells C1-C6")
    for cleaning_num in range(1, 7):  # C1, C2, C3, C4, C5, C6
        cleaning_well_id = f"C{cleaning_num}"
        cleaning_well_xy = get_well_xy(cleaning_well_id)
        
        print(f"    Cleaning in well {cleaning_well_id}")
        zm.moveToZInUM(safe_z)
        s.moveToPos(cleaning_well_xy[0], cleaning_well_xy[1])
        zm.moveToZInUM(in_well_z)
        time.sleep(1)  # Rinse time
        zm.moveToZInUM(safe_z)
    
    print(f"Solution {solution_num} complete!\n")

4/3/2026 progress: idea works now but sequence seems wrong. it goes from A1 to channel to A1 to cleaning to Channel. Fix this 

4/6/2026 progress: sequence is right, but it goes from C6 to C1. 

4/10/2026 progress: finished the sequence, work on the UI next.

4/17/2026 TODO: test the increase in_well_z height on UI. UI working now. figure out if the user abort after solution 2, how to start from solution 3 if accidentally or intentionally closed the popup prompts??

4/20/2026 TODO: single button for washing steps in increasing depth, can use the same C1-C6 wells (rinse six times) (DONE)
- make six solutions to be variable (DONE)
- make ABORT button work (DONE)

In [15]:
# # 4. Full sequence with prompts (for GUI integration)
# def run_full_experiment():
#     for solution_num in range(1, 7):  # 1 to 6
#         # In GUI, this would be a dialog box
#         input(f"Load solution {solution_num} into well A{solution_num} and cleaning into C{solution_num}. Press Enter when ready...")
        
#         print_one_solution(solution_num)

#         print(f"Solution {solution_num} dispensing complete!\n")

#     print("All solutions dispensed!")

# # 5. Test in notebook
# run_full_experiment()

solution_num = 3
print_one_solution(solution_num)

Solution 3: Will print to channels [3, 9, 15, 21, 27, 33, 39]
  Processing channel 3
    Aspirating from well A3
    Dispensing to channel 3
  Processing channel 9
    Aspirating from well A3
    Dispensing to channel 9
  Processing channel 15
    Aspirating from well A3
    Dispensing to channel 15
  Processing channel 21
    Aspirating from well A3
    Dispensing to channel 21
  Processing channel 27
    Aspirating from well A3
    Dispensing to channel 27
  Processing channel 33
    Aspirating from well A3
    Dispensing to channel 33
  Processing channel 39
    Aspirating from well A3
    Dispensing to channel 39
Solution 3: Cleaning in wells C1-C6
    Cleaning in well C1
    Cleaning in well C2
    Cleaning in well C3
    Cleaning in well C4
    Cleaning in well C5
    Cleaning in well C6
Solution 3 complete!



In [40]:
# Verify channel patterns
for i in range(1, 7):
    print(f"Solution {i}: {get_channel_pattern(i)}")

# Verify well positions
for well in ["A1", "A6", "C1", "C6"]:
    print(f"{well}: {get_well_xy(well)}")

Solution 1: [1, 7, 13, 19, 25, 31, 37]
Solution 2: [2, 8, 14, 20, 26, 32, 38]
Solution 3: [3, 9, 15, 21, 27, 33, 39]
Solution 4: [4, 10, 16, 22, 28, 34, 40]
Solution 5: [5, 11, 17, 23, 29, 35]
Solution 6: [6, 12, 18, 24, 30, 36]
A1: (-102318, -5778)
A6: (-79818, -5778)
C1: (-102318, 3222)
C6: (-79818, 3222)
